<a href="https://colab.research.google.com/github/raffson123/DC37_DataAnalytics/blob/main/eda_template_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploratory Data Analysis (EDA) Template

This notebook provides a reusable template for performing EDA on any tabular dataset.

**How to use this notebook:**
1. Upload your CSV file to Colab or mount Google Drive.
2. Update the file path in the data loading cell.
3. (Optional) Set your target variable (for classification/regression).
4. Run cells in order and review outputs.

In [ ]:
# Basic imports
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# Plot settings (optional)
plt.style.use('default')
sns.set(style='whitegrid')

# Show versions (useful when sharing notebooks)
import sys
print('Python version:', sys.version)
print('Pandas version:', pd.__version__)
print('Seaborn version:', sns.__version__)

## 1. Load Dataset

Update the file path or use Google Drive upload.

- If file is local: upload in left sidebar → Files → copy path.
- If file is in Drive: mount Drive and use that path.

In [ ]:
# OPTION A: Local upload in Colab (run this cell and use the upload button)
from google.colab import files

uploaded = files.upload()  # Choose your CSV file
file_name = list(uploaded.keys())[0]
print('Uploaded file:', file_name)

# Read the uploaded CSV
df = pd.read_csv(file_name)

# If you already know the filename/path, you can skip upload and do:
# df = pd.read_csv('your_file.csv')

## 2. Quick Data Overview

In this section we:
- Look at the first and last few rows
- Check shape (rows, columns)
- Get column names and data types
- Get high-level info and basic statistics

In [ ]:
# Peek at the data
print('First 5 rows:')
display(df.head())

print('Last 5 rows:')
display(df.tail())

# Shape of dataset
print('Shape (rows, columns):', df.shape)

# Column names
print('\nColumn names:')
print(df.columns.tolist())

# Data types
print('\nData types:')
print(df.dtypes)

# Info (non-null counts, dtypes)
print('\nDataFrame info:')
df.info()

# Basic numeric stats
print('\nDescriptive statistics (numeric columns):')
display(df.describe())

# Basic stats for categorical columns
print('\nDescriptive statistics (categorical columns):')
display(df.describe(include='object'))

# Memory usage
print('\nMemory usage (bytes):')
print(df.memory_usage(deep=True))

## 3. Missing Values Analysis

We check:
- Count of missing values
- Percentage of missing values
- A simple visual using a heatmap

In [ ]:
# Count of missing values
print('Missing values per column (count):')
missing_count = df.isnull().sum()
display(missing_count)

# Percentage of missing values
print('\nMissing values per column (percentage):')
missing_pct = df.isnull().mean() * 100
display(missing_pct)

# Simple table sorted by percentage
print('\nColumns sorted by missing percentage:')
display(missing_pct.sort_values(ascending=False))

# Visual: missingness heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False)
plt.title('Missing Values Heatmap')
plt.show()

## 4. Duplicate Rows

We:
- Count duplicate rows
- Show them (if any)
- Optionally create a de-duplicated version

In [ ]:
# Number of duplicate rows
num_duplicates = df.duplicated().sum()
print('Number of duplicate rows:', num_duplicates)

# Show duplicate rows (if any)
if num_duplicates > 0:
    print('\nDuplicate rows:')
    display(df[df.duplicated()])
else:
    print('No duplicate rows found.')

# Create a version without duplicates (if you want to use it later)
df_no_dupes = df.drop_duplicates().copy()
print('Shape after removing duplicates:', df_no_dupes.shape)

## 5. Separate Numeric and Categorical Columns

We identify:
- Numeric columns (int/float)
- Categorical columns (object/category)

This helps us run appropriate plots and stats later.

In [ ]:
# Numeric columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print('Numeric columns:', num_cols)

# Categorical columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print('Categorical columns:', cat_cols)

## 6. Univariate Analysis

### 6.1 Numeric Features
We look at:
- Descriptive statistics
- Histograms

### 6.2 Categorical Features
We look at:
- Value counts
- Bar plots for top categories

In [ ]:
# Descriptive statistics (transposed for readability)
print('Numeric feature statistics:')
display(df[num_cols].describe().T)

# Histograms for all numeric columns
df[num_cols].hist(figsize=(12, 10), bins=30)
plt.suptitle('Histograms of Numeric Columns', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Value counts for each categorical column
for col in cat_cols:
    print(f"\nColumn: {col}")
    display(df[col].value_counts(dropna=False))
    display((df[col].value_counts(normalize=True, dropna=False) * 100).rename('percentage'))

# Bar plots for top categories (up to 10)
for col in cat_cols:
    plt.figure(figsize=(6, 4))
    df[col].value_counts(dropna=False).head(10).plot(kind='bar')
    plt.title(f'Top categories in {col}')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 7. Target Variable (Optional)

If you have a **target column** (e.g., `stroke`, `churn`, `label`), set it below.

We will:
- Check target distribution
- Use it later for bivariate analysis

In [ ]:
# Set your target column name here (if you have one)
target_col = None  # e.g., 'stroke', 'churn', 'label'

if target_col is not None and target_col in df.columns:
    print('Target column:', target_col)
    print('\nTarget value counts:')
    display(df[target_col].value_counts())
    print('\nTarget value percentages:')
    display(df[target_col].value_counts(normalize=True) * 100)

    plt.figure(figsize=(6,4))
    sns.countplot(x=target_col, data=df)
    plt.title(f'Distribution of {target_col}')
    plt.show()
else:
    print('No valid target column set. Set target_col to a column name to analyze target.')

## 8. Bivariate Analysis

If a target is defined:

- **Categorical vs Target:** countplots
- **Numeric vs Target:** boxplots and/or KDE plots

If no target, you can skip or adapt to your use case.

In [ ]:
if target_col is not None and target_col in df.columns:
    cat_cols_ex_target = [c for c in cat_cols if c != target_col]

    for col in cat_cols_ex_target:
        plt.figure(figsize=(6, 4))
        sns.countplot(x=col, hue=target_col, data=df)
        plt.title(f'{col} by {target_col}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No target column set. Skipping categorical vs target analysis.')

In [ ]:
if target_col is not None and target_col in df.columns:
    for col in num_cols:
        if col == target_col:
            continue

        plt.figure(figsize=(6, 4))
        sns.boxplot(x=target_col, y=col, data=df)
        plt.title(f'{col} by {target_col}')
        plt.tight_layout()
        plt.show()

        # Optional KDE plot
        plt.figure(figsize=(6, 4))
        sns.kdeplot(data=df, x=col, hue=target_col, common_norm=False)
        plt.title(f'Distribution of {col} by {target_col}')
        plt.tight_layout()
        plt.show()
else:
    print('No target column set. Skipping numeric vs target analysis.')

## 9. Correlation Analysis (Numeric)

We:
- Compute correlation matrix for numeric columns
- Visualize using a heatmap
- (If numeric target) check strongest correlations with target

In [ ]:
# Compute correlation matrix for numeric columns
corr = df[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix of Numeric Features')
plt.show()

# If target is numeric, show correlations with target
if target_col is not None and target_col in num_cols:
    print(f"Correlations with target '{target_col}':")
    display(corr[target_col].sort_values(ascending=False))

## 10. Outlier Checks (Simple IQR Method)

We:
- Use boxplots to see outliers visually
- Use IQR rule to detect outliers for each numeric column

In [ ]:
# Boxplots for numeric columns
for col in num_cols:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot of {col}')
    plt.tight_layout()
    plt.show()

# IQR-based outlier detection summary
outlier_summary = {}

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    outlier_summary[col] = outliers.shape[0]

print('Number of outliers per numeric column (IQR method):')
display(pd.Series(outlier_summary).sort_values(ascending=False))

## 11. (Optional) Date/Time Features

If you have datetime columns, convert and extract useful parts like year, month, day of week, etc.

In [ ]:
# Example: If you know a column is a date
date_col = None  # e.g., 'order_date'

if date_col is not None and date_col in df.columns:
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    df['year'] = df[date_col].dt.year
    df['month'] = df[date_col].dt.month
    df['day'] = df[date_col].dt.day
    df['dayofweek'] = df[date_col].dt.dayofweek

    print(df[[date_col, 'year', 'month', 'day', 'dayofweek']].head())
else:
    print('No date_col set or column not found. Skipping datetime feature extraction.')

## 12. GroupBy & Aggregations (Quick Patterns)

Use this section to find patterns grouped by:
- Target
- Key categorical columns (e.g., gender, region, department)

In [ ]:
# Example 1: Mean of numeric columns by target (if available)
if target_col is not None and target_col in df.columns:
    print(f"Mean numeric values grouped by target '{target_col}':")
    display(df.groupby(target_col)[num_cols].mean())

# Example 2: Category vs target counts
example_cat = None  # e.g., 'gender'

if example_cat is not None and example_cat in df.columns and target_col in df.columns:
    print(f"\nCounts of {example_cat} by {target_col}:")
    display(df.groupby([example_cat, target_col]).size().unstack(fill_value=0))

# Example 3: Aggregation for a chosen numeric column
agg_cat = None      # e.g., 'gender'
agg_num_col = None  # e.g., 'age'

if agg_cat is not None and agg_cat in df.columns and agg_num_col in df.columns:
    print(f"\nAggregations of {agg_num_col} by {agg_cat}:")
    display(df.groupby(agg_cat)[agg_num_col].agg(['count', 'mean', 'min', 'max']))

## 13. Save Cleaned / EDA-Ready Data (Optional)

After cleaning (handling missing values, removing duplicates, etc.), you can save the dataset.

Update `df_clean` to your final cleaned DataFrame.

In [ ]:
# For now we just copy df; in real use, modify df first (cleaning) and then save
df_clean = df.copy()

output_file = 'cleaned_data.csv'
df_clean.to_csv(output_file, index=False)
print(f'Cleaned data saved to {output_file}')